In [2]:
import cv2
import os
import numpy as np
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import threading
import time

In [3]:
def build_video(frames_folder, output_path, is_mask, fps=30):
    images = os.listdir(frames_folder)
    images.sort()
    first_image_path = os.path.join(frames_folder, images[0])
    first_frame = cv2.imread(first_image_path)
    height, width = first_frame.shape[0], first_frame.shape[1]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for idx, image in enumerate(images):
        image_path = os.path.join(frames_folder, image)
        
        if is_mask:
            mask = np.array(Image.open(image_path))
            
            if idx == 0:
                print(f"Уникальные значения в маске: {np.unique(mask)}")
                print(f"Форма маски: {mask.shape}")

            img_original_path = image_path.replace('Annotations', 'JPEGImages').replace('.png', '.jpg')
            img_original = cv2.imread(img_original_path)
            frame = apply_mask_to_frame(img_original, mask)
        else:
            frame = cv2.imread(image_path)
            
        video_writer.write(frame)
    
    video_writer.release()
    print(f'video saved to {output_path}')
    return output_path

In [4]:
def apply_mask_to_frame(frame, mask):
    binary_mask = (mask == 1).astype(np.uint8) * 255
    result = frame.copy()
    result[binary_mask == 255] = [255, 255, 255]
    
    return result

In [5]:
class VideoPlayer:
    def __init__(self, root, original_video_path, masked_video_path):
        self.root = root
        self.root.title("Параллельный видеоплеер")
        
        # Открываем оба видео
        self.cap_orig = cv2.VideoCapture(original_video_path)
        self.cap_mask = cv2.VideoCapture(masked_video_path)
        
        self.fps = int(self.cap_orig.get(cv2.CAP_PROP_FPS))
        self.total_frames = int(self.cap_orig.get(cv2.CAP_PROP_FRAME_COUNT))
        self.current_frame = 0
        self.is_playing = False
        
        # UI: два холста для видео
        self.canvas_orig = tk.Canvas(root, width=400, height=300, bg='black')
        self.canvas_orig.grid(row=0, column=0, padx=5, pady=5)
        
        self.canvas_mask = tk.Canvas(root, width=400, height=300, bg='black')
        self.canvas_mask.grid(row=0, column=1, padx=5, pady=5)
        
        # Слайдер
        self.slider = ttk.Scale(root, from_=0, to=self.total_frames-1, orient='horizontal')
        self.slider.grid(row=1, column=0, columnspan=2, sticky='ew', padx=10)
        self.slider.bind("<ButtonRelease-1>", self.slider_moved)
        
        # Кнопки
        self.btn_play = tk.Button(root, text="▶ Воспроизвести", command=self.play_pause)
        self.btn_play.grid(row=2, column=0, pady=5)
        
        self.btn_stop = tk.Button(root, text="⏹ Стоп", command=self.stop)
        self.btn_stop.grid(row=2, column=1, pady=5)
        
        # Показываем первый кадр
        self.update_frame(0)
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
    
    def get_frame(self, cap, frame_number):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        if ret:
            # Конвертируем BGR (OpenCV) в RGB (PIL/Tkinter)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            img = img.resize((400, 300), Image.Resampling.LANCZOS)
            return ImageTk.PhotoImage(img)
        return None
    
    def update_frame(self, frame_idx):
        # Обновляем оба видео на одном кадре
        img_orig = self.get_frame(self.cap_orig, frame_idx)
        img_mask = self.get_frame(self.cap_mask, frame_idx)
        
        if img_orig and img_mask:
            self.canvas_orig.create_image(0, 0, anchor='nw', image=img_orig)
            self.canvas_mask.create_image(0, 0, anchor='nw', image=img_mask)
            # Сохраняем ссылки, чтобы изображения не удалил сборщик мусора
            self.current_img_orig = img_orig
            self.current_img_mask = img_mask
        
        self.slider.set(frame_idx)
        self.current_frame = frame_idx
    
    def play_pause(self):
        if self.is_playing:
            self.is_playing = False
            self.btn_play.config(text="▶ Воспроизвести")
        else:
            self.is_playing = True
            self.btn_play.config(text="⏸ Пауза")
            self.play_loop()
    
    def play_loop(self):
        if not self.is_playing:
            return
        
        next_frame = self.current_frame + 1
        if next_frame >= self.total_frames:
            self.stop()
            return
        
        self.update_frame(next_frame)
        # Задержка между кадрами в миллисекундах
        delay = int(1000 / self.fps)
        self.root.after(delay, self.play_loop)
    
    def stop(self):
        self.is_playing = False
        self.btn_play.config(text="▶ Воспроизвести")
        self.update_frame(0)
    
    def slider_moved(self, event):
        if self.is_playing:
            self.is_playing = False
            self.btn_play.config(text="▶ Воспроизвести")
        frame_idx = int(self.slider.get())
        self.update_frame(frame_idx)
    
    def on_close(self):
        self.cap_orig.release()
        self.cap_mask.release()
        self.root.destroy()

In [6]:
build_video('../data/YoutubeVos/valid/JPEGImages/0a49f5265b', 'output_original.mp4', is_mask=False)
build_video('../data/YoutubeVos/valid/Annotations/0a49f5265b', 'output_masked.mp4', is_mask=True)
root = tk.Tk()
player = VideoPlayer(root, 'output_original.mp4', 'output_masked.mp4')
root.mainloop()

video saved to output_original.mp4
Уникальные значения в маске: [0 1 2 3]
Форма маски: (720, 1280)
video saved to output_masked.mp4


In [7]:
np.unique(cv2.imread('../data/YoutubeVos/valid/Annotations/0a49f5265b/00000.png'))

array([  0,  87,  95,  99, 103, 145, 200, 236, 249, 250], dtype=uint8)

In [8]:
cv2.imread('../data/YoutubeVos/valid/Annotations/0a49f5265b/00000.png').shape

(720, 1280, 3)

In [9]:
np.unique(Image.open('../data/YoutubeVos/valid/Annotations/0a49f5265b/00000.png'))


array([0, 1, 2, 3], dtype=uint8)

In [10]:
np.array(Image.open('../data/YoutubeVos/valid/Annotations/0a49f5265b/00000.png')).shape

(720, 1280)